<a href="https://colab.research.google.com/github/abhay2147/Multimodal_Anomaly_Detection/blob/main/multimodal_anomaly_detection_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install -q transformers datasets torch scikit-learn pandas matplotlib seaborn shap
print("✅ Dependencies installed")

In [ ]:
import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, f1_score
)
from transformers import DistilBertTokenizer, DistilBertModel
from collections import Counter

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Data Acquisition ──────────────────────────────────────────────────────────
# Option A: Download UNSW-NB15 directly (CSV subset available via direct link)
# Option B: The cell below generates a high-fidelity synthetic stand-in so the
#           notebook runs end-to-end in Colab without manual uploads.
# Set USE_SYNTHETIC = False and provide CSV_PATH to use real data.

USE_SYNTHETIC = True   # ← flip to False when using real UNSW-NB15 CSV
CSV_PATH      = '/content/UNSW_NB15_training-set.csv'

ATTACK_CLASSES = ['Normal', 'Fuzzers', 'Analysis', 'Backdoors',
                  'DoS', 'Exploits', 'Reconnaissance', 'Worms']

# Representative log templates per class (HDFS/BGL-style)
LOG_TEMPLATES = {
    'Normal'         : [
        "INFO dfs.DataNode$DataXceiver: Received block blk_{id} src: /{ip}",
        "INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /user/data/part-{id}",
        "INFO mapred.TaskTracker: Done with task tracker: tracker_node{id}",
    ],
    'Fuzzers'        : [
        "WARN security.AuditLogger: Malformed packet received from {ip} port {port} - possible fuzzing",
        "ERROR kernel: buffer overflow detected in recv() from {ip}:{port}",
        "WARN net.filter: Unexpected protocol mutation in stream from {ip}",
    ],
    'Analysis'       : [
        "WARN dfs.FSNamesystem: Unauthorized directory listing attempted from {ip}",
        "INFO audit: Multiple sequential port probes from host {ip} detected",
        "WARN sshd: Failed publickey for root from {ip} port {port} ssh2",
    ],
    'Backdoors'      : [
        "ERROR kernel: Unexpected outbound connection on port 4444 to {ip}",
        "ALERT ids: Reverse shell pattern detected in payload from {ip}:{port}",
        "CRIT security: Unauthorized cron job modification detected by uid=0",
    ],
    'DoS'            : [
        "ERROR net: SYN flood detected from {ip} - dropping connections",
        "CRIT kernel: ICMP flood: {n} packets/sec from {ip} - rate limiting engaged",
        "WARN apache: Connection table exhausted - possible DoS from {ip}",
    ],
    'Exploits'       : [
        "CRIT ids: SQL injection attempt in HTTP payload from {ip}:{port}",
        "ERROR apache: Shellcode pattern found in GET request from {ip}",
        "ALERT security: CVE-2021-44228 Log4Shell exploit attempt from {ip}",
    ],
    'Reconnaissance' : [
        "WARN firewall: Port scan detected from {ip} - {n} ports probed in 10s",
        "INFO snort: NMAP OS detection attempt from {ip}",
        "WARN sshd: {n} failed login attempts from {ip} in 60 seconds",
    ],
    'Worms'          : [
        "CRIT antivirus: Worm propagation pattern detected originating from {ip}",
        "ERROR net: Self-replicating traffic to {n} unique subnets in 5s",
        "ALERT ids: Conficker-like DNS query storm from host {ip}",
    ],
}

def generate_log(label):
    tmpl = random.choice(LOG_TEMPLATES[label])
    return tmpl.format(
        ip=f"{random.randint(1,254)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}",
        port=random.randint(1024, 65535),
        id=random.randint(100000, 999999),
        n=random.randint(100, 50000),
    )

if USE_SYNTHETIC:
    print("⚠️  Generating synthetic UNSW-NB15-style data...")
    N = 12000
    # Class distribution mimicking real UNSW-NB15 imbalance
    class_weights = [0.45, 0.12, 0.07, 0.03, 0.10, 0.12, 0.09, 0.02]
    labels = random.choices(ATTACK_CLASSES, weights=class_weights, k=N)

    PROTO_MAP = {'Normal':'tcp','Fuzzers':'udp','Analysis':'tcp',
                 'Backdoors':'tcp','DoS':'icmp','Exploits':'tcp',
                 'Reconnaissance':'udp','Worms':'tcp'}
    STATE_MAP = {'Normal':'FIN','Fuzzers':'INT','Analysis':'CON',
                 'Backdoors':'FIN','DoS':'INT','Exploits':'CON',
                 'Reconnaissance':'CON','Worms':'FIN'}

    rows = []
    for lbl in labels:
        is_attack = int(lbl != 'Normal')
        rows.append({
            'dur'      : np.random.exponential(0.5) if not is_attack else np.random.exponential(5),
            'sbytes'   : np.random.randint(50, 5000) if not is_attack else np.random.randint(1000,100000),
            'dbytes'   : np.random.randint(50, 5000) if not is_attack else np.random.randint(100, 20000),
            'sttl'     : random.choice([64, 128, 255]),
            'dttl'     : random.choice([64, 128, 255]),
            'sloss'    : np.random.poisson(0.1) if not is_attack else np.random.poisson(3),
            'dloss'    : np.random.poisson(0.1) if not is_attack else np.random.poisson(2),
            'sload'    : np.random.exponential(100),
            'dload'    : np.random.exponential(80),
            'spkts'    : np.random.randint(1, 50) if not is_attack else np.random.randint(50, 5000),
            'dpkts'    : np.random.randint(1, 40) if not is_attack else np.random.randint(10, 2000),
            'proto'    : PROTO_MAP[lbl],
            'state'    : STATE_MAP[lbl],
            'attack_cat': lbl,
        })

    df = pd.DataFrame(rows)
    df['log_text'] = [generate_log(lbl) for lbl in labels]
    print(f"✅ Synthetic dataset: {len(df):,} samples")
    print(dict(Counter(labels)))
else:
    df = pd.read_csv(CSV_PATH)
    # Assume UNSW-NB15 column 'attack_cat' exists; add placeholder logs
    df['attack_cat'] = df['attack_cat'].fillna('Normal').str.strip()
    df['log_text']   = df['attack_cat'].apply(generate_log)
    print(f"✅ Loaded real dataset: {len(df):,} samples")

df.head(3)

In [ ]:
# ── 2.1 Tabular Pipeline ──────────────────────────────────────────────────────
NUMERIC_COLS = ['dur','sbytes','dbytes','sttl','dttl','sloss','dloss',
                'sload','dload','spkts','dpkts']
CAT_COLS     = ['proto','state']
LABEL_COL    = 'attack_cat'

# Drop any rows with nulls in key columns
df = df.dropna(subset=NUMERIC_COLS + CAT_COLS + [LABEL_COL]).reset_index(drop=True)

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df[LABEL_COL])
NUM_CLASSES  = len(le.classes_)
print(f"Classes ({NUM_CLASSES}): {list(le.classes_)}")

# One-hot encode categoricals
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = ohe.fit_transform(df[CAT_COLS])
cat_df = pd.DataFrame(cat_encoded, columns=ohe.get_feature_names_out(CAT_COLS))

# Scale numerics
scaler = StandardScaler()
num_scaled = scaler.fit_transform(df[NUMERIC_COLS])
num_df = pd.DataFrame(num_scaled, columns=NUMERIC_COLS)

# Final feature matrix
X_tab   = np.concatenate([num_scaled, cat_encoded], axis=1).astype(np.float32)
y_all   = df['label'].values
logs_all = df['log_text'].values

N_FEATURES = X_tab.shape[1]
print(f"✅ Tabular feature dim: {N_FEATURES}")

In [ ]:
# ── 2.2 Textual Pipeline ─────────────────────────────────────────────────────
MAX_LEN   = 128
BERT_NAME = 'distilbert-base-uncased'

print(f"Loading DistilBERT tokenizer: {BERT_NAME}")
tokenizer = DistilBertTokenizer.from_pretrained(BERT_NAME)

def tokenize_logs(texts, max_len=MAX_LEN):
    return tokenizer(
        list(texts),
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

# Quick sanity check
sample_enc = tokenize_logs(logs_all[:2])
print(f"✅ input_ids shape: {sample_enc['input_ids'].shape}")
print(f"   attention_mask shape: {sample_enc['attention_mask'].shape}")

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class NetworkLogDataset(Dataset):
    def __init__(self, tab_feats, input_ids, attn_masks, labels):
        self.tab    = torch.tensor(tab_feats, dtype=torch.float32)
        self.ids    = input_ids
        self.masks  = attn_masks
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'tab'   : self.tab[idx],
            'ids'   : self.ids[idx],
            'masks' : self.masks[idx],
            'label' : self.labels[idx],
        }

In [ ]:
# ── Model Definition ──────────────────────────────────────────────────────────
class MultimodalAnomalyDetector(nn.Module):
    """Late-fusion: MLP (numeric) + DistilBERT (text) → classifier."""

    def __init__(self, n_num_features: int, n_classes: int,
                 mlp_hidden: int = 128, bert_name: str = BERT_NAME,
                 dropout: float = 0.2):
        super().__init__()

        # ── Numeric branch (3-layer MLP) ──────────────────────────────────
        self.mlp = nn.Sequential(
            nn.Linear(n_num_features, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2),
            nn.ReLU(),
        )
        mlp_out_dim = mlp_hidden // 2

        # ── Textual branch (DistilBERT) ───────────────────────────────────
        self.bert    = DistilBertModel.from_pretrained(bert_name)
        bert_out_dim = self.bert.config.hidden_size  # 768

        # Start with BERT frozen (will be unfrozen after N epochs)
        self.freeze_bert()

        # ── Fusion + classifier ───────────────────────────────────────────
        fusion_dim = mlp_out_dim + bert_out_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def freeze_bert(self):
        for p in self.bert.parameters():
            p.requires_grad = False

    def unfreeze_bert(self):
        for p in self.bert.parameters():
            p.requires_grad = True

    def forward(self, tab, input_ids, attention_mask):
        # Numeric branch
        x_num = self.mlp(tab)                                        # (B, mlp_out)

        # Textual branch – use [CLS] token embedding
        bert_out = self.bert(input_ids=input_ids,
                             attention_mask=attention_mask)
        x_txt = bert_out.last_hidden_state[:, 0, :]                  # (B, 768)

        # Fusion
        x = torch.cat([x_num, x_txt], dim=1)                        # (B, fusion_dim)
        return self.classifier(x)                                    # (B, n_classes)


# Baseline: numeric-only model for comparative analysis
# forward() accepts *args/**kwargs so it is drop-in compatible with
# eval_epoch(), which always calls model(tab, input_ids, attention_mask).
class NumericOnlyDetector(nn.Module):
    def __init__(self, n_features: int, n_classes: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden),     nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),nn.ReLU(),
            nn.Linear(hidden // 2, n_classes)
        )
    def forward(self, tab, *args, **kwargs):  # ignore ids/masks
        return self.net(tab)


print("✅ Model classes defined")

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
def compute_class_weights(labels, n_classes):
    """Inverse-frequency class weights for weighted CrossEntropy."""
    counts = np.bincount(labels, minlength=n_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    weights /= weights.sum()
    return torch.tensor(weights * n_classes, dtype=torch.float32)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, n = 0., 0, 0
    for batch in loader:
        tab   = batch['tab'].to(device)
        ids   = batch['ids'].to(device)
        masks = batch['masks'].to(device)
        labs  = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(tab, ids, masks)
        loss   = criterion(logits, labs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(labs)
        correct    += (logits.argmax(1) == labs).sum().item()
        n          += len(labs)
    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labs, all_probs = 0., [], [], []
    for batch in loader:
        tab   = batch['tab'].to(device)
        ids   = batch['ids'].to(device)
        masks = batch['masks'].to(device)
        labs  = batch['label'].to(device)

        logits = model(tab, ids, masks)
        loss   = criterion(logits, labs)
        probs  = torch.softmax(logits, dim=1)

        total_loss += loss.item() * len(labs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labs.extend(labs.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    n = len(all_labs)
    macro_f1 = f1_score(all_labs, all_preds, average='macro', zero_division=0)
    return total_loss / n, macro_f1, np.array(all_preds), np.array(all_labs), np.array(all_probs)

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
LR            = 2e-5
BATCH_SIZE    = 32
EPOCHS        = 8
UNFREEZE_EPOCH = 5   # unfreeze DistilBERT after this epoch
N_FOLDS       = 5

# Pre-tokenise ALL logs once (saves repeated tokenisation inside fold loop)
print("Tokenising all logs (this may take ~30s)...")
all_encodings = tokenize_logs(logs_all)
ALL_IDS   = all_encodings['input_ids']
ALL_MASKS = all_encodings['attention_mask']
print(f"✅ Tokenisation complete. Tensor shape: {ALL_IDS.shape}")

In [ ]:
# ── 5-Fold Cross-Validation Training Loop ────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_results = []
history      = {'train_loss': [], 'val_loss': [], 'val_f1': []}

# Only run fold 1 for quick demo; set to N_FOLDS for full CV
MAX_FOLDS_TO_RUN = 1  # ← change to N_FOLDS for full 5-fold

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tab, y_all)):
    if fold >= MAX_FOLDS_TO_RUN:
        break
    print(f"\n{'='*55}")
    print(f"  FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*55}")

    # Build datasets
    tr_ds = NetworkLogDataset(X_tab[tr_idx], ALL_IDS[tr_idx], ALL_MASKS[tr_idx], y_all[tr_idx])
    va_ds = NetworkLogDataset(X_tab[va_idx], ALL_IDS[va_idx], ALL_MASKS[va_idx], y_all[va_idx])
    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    va_dl = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Weighted loss
    cw        = compute_class_weights(y_all[tr_idx], NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=cw)

    # Model & optimizer
    model = MultimodalAnomalyDetector(N_FEATURES, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    fold_hist = {'train_loss': [], 'val_loss': [], 'val_f1': []}

    for epoch in range(1, EPOCHS + 1):
        # Unfreeze BERT at the scheduled epoch
        if epoch == UNFREEZE_EPOCH:
            print(f"  [Epoch {epoch}] Unfreezing DistilBERT for fine-tuning...")
            model.unfreeze_bert()
            # Rebuild optimizer to include newly unfrozen BERT params
            optimizer = torch.optim.AdamW(
                [{"params": model.bert.parameters(),       "lr": LR * 0.1},
                 {"params": model.mlp.parameters(),        "lr": LR},
                 {"params": model.classifier.parameters(), "lr": LR}]
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - epoch)

        tr_loss, tr_acc = train_epoch(model, tr_dl, optimizer, criterion, DEVICE)
        va_loss, va_f1, va_preds, va_labs, va_probs = eval_epoch(model, va_dl, criterion, DEVICE)
        scheduler.step()

        fold_hist['train_loss'].append(tr_loss)
        fold_hist['val_loss'].append(va_loss)
        fold_hist['val_f1'].append(va_f1)

        print(f"  Epoch {epoch:02d}/{EPOCHS} | "
              f"Train Loss: {tr_loss:.4f} | Acc: {tr_acc:.3f} | "
              f"Val Loss: {va_loss:.4f} | Val F1: {va_f1:.3f}")

    history['train_loss'].extend(fold_hist['train_loss'])
    history['val_loss'].extend(fold_hist['val_loss'])
    history['val_f1'].extend(fold_hist['val_f1'])
    fold_results.append({'fold': fold+1, 'val_f1': max(fold_hist['val_f1']),
                         'preds': va_preds, 'labs': va_labs, 'probs': va_probs})

print("\n✅ Training complete")

In [ ]:
# ── Learning Curves ───────────────────────────────────────────────────────────
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='orangered')
axes[0].set_title('Loss Curves'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['val_f1'], color='seagreen', marker='o', markersize=4)
axes[1].set_title('Validation Macro F1'); axes[1].set_xlabel('Epoch')
axes[1].grid(alpha=0.3)

plt.suptitle('Multimodal Model Training Dynamics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
best_fold  = max(fold_results, key=lambda x: x['val_f1'])
va_preds   = best_fold['preds']
va_labs    = best_fold['labs']
va_probs   = best_fold['probs']
class_names = le.classes_

cm = confusion_matrix(va_labs, va_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted Class', fontsize=12)
ax.set_ylabel('True Class',      fontsize=12)
ax.set_title('Confusion Matrix (Normalised) — Multimodal Model', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n", classification_report(va_labs, va_preds,
                                   target_names=class_names, zero_division=0))

In [ ]:
# ── PR-AUC & FPR per class ────────────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve, auc
from sklearn.preprocessing import label_binarize

y_bin = label_binarize(va_labs, classes=range(NUM_CLASSES))

pr_aucs, fprs = [], []
for i, cls in enumerate(class_names):
    if y_bin[:, i].sum() == 0:
        continue
    prec, rec, _ = precision_recall_curve(y_bin[:, i], va_probs[:, i])
    pr_aucs.append((cls, auc(rec, prec)))

    # FPR = FP / (FP + TN)
    tp = ((va_preds == i) & (va_labs == i)).sum()
    fp = ((va_preds == i) & (va_labs != i)).sum()
    fn = ((va_preds != i) & (va_labs == i)).sum()
    tn = ((va_preds != i) & (va_labs != i)).sum()
    fpr = fp / (fp + tn + 1e-9)
    fprs.append((cls, fpr))

# Tabulate
metrics_df = pd.DataFrame(pr_aucs, columns=['Class','PR-AUC']).merge(
    pd.DataFrame(fprs, columns=['Class','FPR']), on='Class')
metrics_df['Macro-F1'] = [
    f1_score(va_labs, va_preds, labels=[i], average='macro', zero_division=0)
    for i in range(len(class_names))
]
print(metrics_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(metrics_df))
w = 0.3
ax.bar(x - w, metrics_df['PR-AUC'],   w, label='PR-AUC',  color='steelblue')
ax.bar(x,     metrics_df['Macro-F1'], w, label='F1',       color='seagreen')
ax.bar(x + w, metrics_df['FPR'],      w, label='FPR',      color='tomato')
ax.set_xticks(x); ax.set_xticklabels(metrics_df['Class'], rotation=25, ha='right')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('Per-Class PR-AUC, F1, and FPR — Multimodal Model', fontweight='bold')
plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Baseline Comparison: Numeric-Only vs Multimodal ───────────────────────────
print("Training Numeric-Only baseline...")
tr_idx_b, va_idx_b = list(skf.split(X_tab, y_all))[0]   # same fold 1 split

class _TabDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        # ids/masks are dummy placeholders — NumericOnlyDetector ignores them via *args.
        # Shape must be (MAX_LEN,) so DataLoader can stack them into (B, MAX_LEN) batches.
        return {
            'tab'  : self.X[i],
            'ids'  : torch.zeros(MAX_LEN, dtype=torch.long),
            'masks': torch.zeros(MAX_LEN, dtype=torch.long),
            'label': self.y[i],
        }

base_model = NumericOnlyDetector(N_FEATURES, NUM_CLASSES).to(DEVICE)
base_crit  = nn.CrossEntropyLoss(weight=compute_class_weights(y_all[tr_idx_b], NUM_CLASSES).to(DEVICE))
base_opt   = torch.optim.AdamW(base_model.parameters(), lr=1e-3)
b_tr_dl    = DataLoader(_TabDS(X_tab[tr_idx_b], y_all[tr_idx_b]), batch_size=64, shuffle=True)
b_va_dl    = DataLoader(_TabDS(X_tab[va_idx_b], y_all[va_idx_b]), batch_size=64)

b_f1s = []
for ep in range(EPOCHS):
    base_model.train()
    for batch in b_tr_dl:
        base_opt.zero_grad()
        loss = base_crit(base_model(batch['tab'].to(DEVICE)), batch['label'].to(DEVICE))
        loss.backward(); base_opt.step()

    _, b_f1, _, _, _ = eval_epoch(base_model, b_va_dl, base_crit, DEVICE)
    b_f1s.append(b_f1)
    print(f"  Epoch {ep+1}/{EPOCHS} | Baseline Val F1: {b_f1:.3f}")

# Summary comparison
mm_f1   = max(fold_results[0]['val_f1'], fold_results[-1]['val_f1'] if len(fold_results)>1 else fold_results[0]['val_f1'])
base_f1 = max(b_f1s)
delta   = mm_f1 - base_f1

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Numeric Only', 'Multimodal\n(+ Logs)'], [base_f1, mm_f1],
               color=['steelblue', 'seagreen'], width=0.5, edgecolor='white')
for bar, val in zip(bars, [base_f1, mm_f1]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.05); ax.set_ylabel('Macro F1'); ax.grid(axis='y', alpha=0.3)
ax.set_title(f'Log Intelligence Boost: Δ F1 = +{delta:.3f}', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n🏆 Multimodal F1: {mm_f1:.3f}  |  Baseline F1: {base_f1:.3f}  |  Gain: +{delta:.3f}")

In [ ]:
# ── SHAP Explainability (Tabular Branch) ──────────────────────────────────────
# We explain the MLP sub-network (numeric features only) via SHAP.
# The text branch uses zero embeddings here so SHAP attribution is purely
# over the tabular feature space — matching what summary_plot expects.
import shap

model.eval()

feature_names = NUMERIC_COLS + list(ohe.get_feature_names_out(CAT_COLS))
N_FEAT = len(feature_names)  # must equal X_tab.shape[1]

# ── Predictor wrapper: returns (n_samples, n_classes) probability array ──────
def mlp_predict(x_np: np.ndarray) -> np.ndarray:
    """Score tabular rows through the MLP branch + classifier.
    Text branch is zeroed out so SHAP attribution stays in tabular space."""
    x_np = np.atleast_2d(x_np).astype(np.float32)
    with torch.no_grad():
        t      = torch.tensor(x_np, dtype=torch.float32).to(DEVICE)
        mlp_out = model.mlp(t)                                  # (B, 64)
        zeros   = torch.zeros(len(t), 768, device=DEVICE)       # null text
        fused   = torch.cat([mlp_out, zeros], dim=1)            # (B, 832)
        logits  = model.classifier(fused)                       # (B, C)
        return torch.softmax(logits, dim=1).cpu().numpy()       # (B, C)

# ── Background & explain sample ───────────────────────────────────────────────
bg_idx      = np.random.choice(tr_idx, size=100, replace=False)
bg_data     = X_tab[bg_idx]          # (100, N_FEAT)
shap_sample = X_tab[va_idx[:50]]     # ( 50, N_FEAT)

# Use PermutationExplainer — faster than KernelExplainer and avoids the
# 3-D array vs list ambiguity that causes the shape AssertionError.
# It returns an Explanation object with .values of shape (50, N_FEAT, C).
print('Running SHAP PermutationExplainer (this takes ~1 min)...')
explainer = shap.PermutationExplainer(mlp_predict, bg_data)
shap_exp  = explainer(shap_sample)   # Explanation: .values (50, N_FEAT, C)

# ── Select target class ───────────────────────────────────────────────────────
target_cls  = list(le.classes_).index('Backdoors') if 'Backdoors' in le.classes_ else 1

# ── Normalise SHAP output → always (n_samples, N_FEAT, n_classes) ────────────
# PermutationExplainer gives Explanation.values of shape (n_samples, N_FEAT, C).
# KernelExplainer gives a Python list of C arrays each (n_samples, N_FEAT).
# Handle both so the cell is robust across SHAP versions.
sv_raw = shap_exp.values
if isinstance(sv_raw, list):
    # list[C] of (n_samples, N_FEAT) → (n_samples, N_FEAT, C)
    sv = np.stack(sv_raw, axis=-1)
elif sv_raw.ndim == 2:
    # single-output: (n_samples, N_FEAT) → (n_samples, N_FEAT, 1)
    sv = sv_raw[:, :, np.newaxis]
else:
    sv = sv_raw  # already (n_samples, N_FEAT, C)

assert sv.shape[1] == N_FEAT, (
    f"SHAP feature dim {sv.shape[1]} != expected {N_FEAT}. Full shape: {sv.shape}"
)
# Clamp in case explainer returned fewer classes than the model (e.g. 1 vs C)
target_cls_safe = min(target_cls, sv.shape[2] - 1)
target_name     = le.classes_[target_cls_safe]       # resolved after shape is known
sv_cls = sv[:, :, target_cls_safe]                   # (50, N_FEAT)

# Build a clean single-class Explanation for beeswarm/summary_plot
bv_raw = shap_exp.base_values
# base_values can be (n_samples,) for single-output or (n_samples, C) for multi
bv_cls = bv_raw[:, target_cls_safe] if bv_raw.ndim == 2 else bv_raw
shap_exp_cls = shap.Explanation(
    values        = sv_cls,
    base_values   = bv_cls,
    data          = shap_sample,
    feature_names = feature_names,
)

# ── Beeswarm plot ─────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_exp_cls, max_display=15, show=False)
plt.title(f'SHAP Feature Importance — Class: "{target_name}"',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Bar chart of mean |SHAP| across all classes ───────────────────────────────
mean_abs_shap = np.abs(sv).mean(axis=(0, 2))         # (N_FEAT,)
top_k = 15
top_idx   = np.argsort(mean_abs_shap)[-top_k:][::-1]
top_names = [feature_names[i] for i in top_idx]
top_vals  = mean_abs_shap[top_idx]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_names[::-1], top_vals[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Mean |SHAP value| (all classes)')
ax.set_title('Global Feature Importance — Tabular Branch', fontweight='bold', fontsize=13)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ SHAP plots saved.')

In [ ]:
# ── Log-Level Token Attribution (Gradient × Input) ───────────────────────────
# Highlights which tokens in a log string drove the classification decision.

def get_token_attribution(model, tokenizer, log_text, tab_vec, device, class_idx):
    """Compute gradient × input attribution for each token in log_text."""
    model.eval()
    enc = tokenizer(log_text, return_tensors='pt',
                    max_length=MAX_LEN, padding='max_length', truncation=True)
    ids   = enc['input_ids'].to(device)
    masks = enc['attention_mask'].to(device)
    tab_t = torch.tensor(tab_vec, dtype=torch.float32).unsqueeze(0).to(device)

    # Embed tokens
    embs = model.bert.embeddings(ids)   # (1, seq_len, 768)
    embs.retain_grad()

    bert_out = model.bert(inputs_embeds=embs, attention_mask=masks)
    cls_emb  = bert_out.last_hidden_state[:, 0, :]
    mlp_out  = model.mlp(tab_t)
    fused    = torch.cat([mlp_out, cls_emb], dim=1)
    logits   = model.classifier(fused)

    logits[0, class_idx].backward()
    grads    = embs.grad.squeeze(0)       # (seq_len, 768)
    embs_det = embs.squeeze(0).detach()
    attrs    = (grads * embs_det).sum(-1).abs().cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(ids.squeeze().cpu().tolist())
    return tokens, attrs


def visualise_token_attribution(tokens, attrs, title='Token Attribution'):
    # Show only non-padding tokens
    valid = [(t, a) for t, a in zip(tokens, attrs)
             if t not in ['[PAD]', '[SEP]']]
    if not valid: return
    toks, vals = zip(*valid)
    vals = np.array(vals)
    vals = vals / (vals.max() + 1e-9)

    fig, ax = plt.subplots(figsize=(max(8, len(toks)*0.45), 2.5))
    colors = plt.cm.Reds(vals)
    ax.bar(range(len(toks)), vals, color=colors, edgecolor='white')
    ax.set_xticks(range(len(toks)))
    ax.set_xticklabels(toks, rotation=45, ha='right', fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Normalised Gradient Attribution')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('token_attribution.png', dpi=150, bbox_inches='tight')
    plt.show()


# Demo on a sample from the validation set
sample_idx  = va_idx[0]
sample_log  = logs_all[sample_idx]
sample_tab  = X_tab[sample_idx]
sample_pred = va_preds[0]
pred_class  = le.classes_[sample_pred]

print(f"Log: {sample_log}")
print(f"Predicted class: {pred_class}")

toks, attrs = get_token_attribution(model, tokenizer, sample_log, sample_tab,
                                    DEVICE, sample_pred)
visualise_token_attribution(toks, attrs,
    title=f'Token Attribution → Predicted: "{pred_class}"')

In [ ]:
# ── Production Inference Function ─────────────────────────────────────────────
@torch.no_grad()
def detect_threat(packet_dict: dict, log_string: str,
                  threshold: float = 0.5) -> dict:
    """
    Classify a single network event.

    Parameters
    ----------
    packet_dict : dict  — raw packet features (keys matching NUMERIC_COLS + CAT_COLS)
    log_string  : str   — raw system log line associated with the event
    threshold   : float — minimum probability to assert a positive detection

    Returns
    -------
    dict with keys:
        predicted_class   : str   — e.g. 'Backdoors'
        confidence        : float — max softmax probability
        is_threat         : bool  — True when class != 'Normal' and conf >= threshold
        class_probabilities: dict — full probability distribution
    """
    model.eval()

    # 1. Tabular preprocessing
    num_vals = np.array([[packet_dict.get(c, 0) for c in NUMERIC_COLS]], dtype=np.float32)
    cat_vals = [[packet_dict.get(c, '') for c in CAT_COLS]]
    num_scaled_inp = scaler.transform(num_vals)
    cat_enc_inp    = ohe.transform(cat_vals)
    tab_t = torch.tensor(
        np.concatenate([num_scaled_inp, cat_enc_inp], axis=1),
        dtype=torch.float32
    ).to(DEVICE)

    # 2. Tokenise log
    enc   = tokenizer(log_string, return_tensors='pt',
                      max_length=MAX_LEN, padding='max_length', truncation=True)
    ids   = enc['input_ids'].to(DEVICE)
    masks = enc['attention_mask'].to(DEVICE)

    # 3. Forward pass
    logits = model(tab_t, ids, masks)
    probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    pred_idx   = int(probs.argmax())
    pred_class = le.classes_[pred_idx]
    confidence = float(probs[pred_idx])

    return {
        'predicted_class'    : pred_class,
        'confidence'         : round(confidence, 4),
        'is_threat'          : pred_class != 'Normal' and confidence >= threshold,
        'class_probabilities': {cls: round(float(p), 4)
                                for cls, p in zip(le.classes_, probs)},
    }


# ── Quick Demo ────────────────────────────────────────────────────────────────
test_packet = {
    'dur': 12.5, 'sbytes': 95000, 'dbytes': 1200, 'sttl': 64, 'dttl': 128,
    'sloss': 5,  'dloss': 1,      'sload': 850,   'dload': 40,
    'spkts': 980,'dpkts': 30,     'proto': 'tcp', 'state': 'FIN'
}
test_log = "CRIT ids: Reverse shell pattern detected in payload from 192.168.1.44:4444"

result = detect_threat(test_packet, test_log)
print("\n🔍 Threat Detection Result:")
print(f"   Predicted Class : {result['predicted_class']}")
print(f"   Confidence      : {result['confidence']:.1%}")
print(f"   Is Threat?      : {'⚠️  YES' if result['is_threat'] else '✅  NO'}")
print("   Class Probabilities:")
for cls, prob in sorted(result['class_probabilities'].items(),
                        key=lambda x: x[1], reverse=True):
    bar = '█' * int(prob * 30)
    print(f"     {cls:<18} {prob:.3f}  {bar}")

In [ ]:
import pickle, json

# Save model weights
torch.save(model.state_dict(), 'multimodal_anomaly_detector.pt')

# Save preprocessors
with open('preprocessors.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'ohe': ohe, 'le': le}, f)

# Save config
config = {
    'n_features'    : int(N_FEATURES),
    'n_classes'     : int(NUM_CLASSES),
    'class_names'   : list(le.classes_),
    'numeric_cols'  : NUMERIC_COLS,
    'cat_cols'      : CAT_COLS,
    'max_len'       : MAX_LEN,
    'bert_name'     : BERT_NAME,
    'best_val_f1'   : round(float(best_fold['val_f1']), 4),
}
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Saved:")
print("   multimodal_anomaly_detector.pt")
print("   preprocessors.pkl")
print("   model_config.json")
print("   training_curves.png")
print("   confusion_matrix.png")
print("   per_class_metrics.png")
print("   model_comparison.png")
print("   shap_summary.png")
print("   token_attribution.png")

# Download all to local machine
try:
    from google.colab import files
    for fname in ['multimodal_anomaly_detector.pt', 'preprocessors.pkl',
                  'model_config.json', 'confusion_matrix.png',
                  'shap_summary.png', 'model_comparison.png']:
        files.download(fname)
except ImportError:
    print("(Not running in Colab — skipping download)")